# Import et préparation du nettoyage — Citiz AURA

Ce notebook charge `regional-data/data/release/citiz_aura_final.csv` dans **`df`**, une copie de travail pandas. Une ligne représente une station et `station_uid` permet de la retrouver lors des jointures.

Utiliser le noyau **Python (IGN / OSM)** de `ign-osm/.venv`, puis exécuter les cellules dans l’ordre. Le CSV source est conservé : aucune écriture ni suppression automatique n’est effectuée.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

current_dir = Path.cwd().resolve()
root = next(
    (directory for directory in [current_dir, *current_dir.parents]
     if (directory / "regional-data/data/release/citiz_aura_final.csv").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Cannot locate regional-data/data/release/citiz_aura_final.csv")

source_path = root / "regional-data/data/release/citiz_aura_final.csv"
# Preserve identifiers and literal text; only empty CSV cells represent missing values.
df_raw = pd.read_csv(
    source_path, encoding="utf-8-sig", sep=",",
    dtype={"station_uid": "string"}, keep_default_na=False, na_values=[""],
).convert_dtypes()
df = df_raw.copy(deep=True)
print(f"Imported {len(df):,} stations and {len(df.columns)} columns")
display(df.head())

Imported 612 stations and 50 columns


,station_uid,station_name,latitude,longitude,irve_count_500m,amenities_count_500m,amenities_count_500m_straight_line,amenities_unreachable_or_unsnapped_500m,bike_parking_count_300m,bike_parking_count_300m_straight_line,...,boundary_coverage_status,city,departement,region,reseau,capacity,num_vehicles_available,is_renting,extracted_at,station_reference_match
0,offre-et-disponibilite-temps-reel-du-service-c...,Lyon 2ème - Parking LPA République,45.760189,4.836602,<NA>,56,323,177,5,36,...,within_region_conservative_check,<NA>,Rhône,Auvergne-Rhône-Alpes,Métropole de Lyon,3,3,True,2026-10-19 14:43:54.580579+00:00,both
1,offre-et-disponibilite-temps-reel-du-service-c...,Lyon 6ème - Bugeaud - Garibaldi,45.766689,4.851268,<NA>,1,326,0,4,61,...,within_region_conservative_check,<NA>,Rhône,Auvergne-Rhône-Alpes,Métropole de Lyon,2,2,True,2026-10-19 14:43:54.580579+00:00,both
2,offre-et-disponibilite-temps-reel-du-service-c...,Sainte-Foy - Place St Luc,45.749405,4.8011,<NA>,9,29,11,1,8,...,within_region_conservative_check,<NA>,Rhône,Auvergne-Rhône-Alpes,Métropole de Lyon,1,1,True,2026-10-19 14:43:54.580579+00:00,both
3,offre-et-disponibilite-temps-reel-du-service-c...,Lyon 2ème - Charité - Auguste Comte,45.754833,4.83118,<NA>,4,262,0,1,53,...,within_region_conservative_check,<NA>,Rhône,Auvergne-Rhône-Alpes,Métropole de Lyon,2,1,True,2026-10-19 14:43:54.580579+00:00,both
4,offre-et-disponibilite-temps-reel-du-service-c...,Lyon 5ème - Quai Pierre Scize,45.766705,4.820361,<NA>,0,42,0,1,25,...,within_region_conservative_check,<NA>,Rhône,Auvergne-Rhône-Alpes,Métropole de Lyon,3,3,True,2026-10-19 14:43:54.580579+00:00,both


## Premier bilan avant nettoyage

Les valeurs manquantes restent inconnues : il ne faut pas les remplacer par zéro. Les colonnes de statut expliquent notamment les indicateurs non calculés. Un zéro dans un comptage OSM indique qu’aucun objet n’a été recensé dans les données traitées, sans garantir son absence sur le terrain.

On distingue les lignes entièrement identiques des identifiants de station répétés. Les doublons sont seulement signalés : leur suppression exige d’abord de comprendre leur origine. Voir aussi `../data/release/DATA_DICTIONARY.md` et `../data/release/SOURCES.md`.

In [2]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_non_null": df.nunique(dropna=True),
}).rename_axis("column").sort_values("missing_count", ascending=False)
display(quality)

print(f"Fully duplicated rows: {df.duplicated().sum()}")
print(f"Missing station identifiers: {df['station_uid'].isna().sum()}")
duplicate_station_mask = df["station_uid"].notna() & df["station_uid"].duplicated(keep=False)
print(f"Rows with repeated station identifiers: {duplicate_station_mask.sum()}")
if duplicate_station_mask.any():
    display(df.loc[duplicate_station_mask].sort_values("station_uid"))

,dtype,missing_count,missing_percent,unique_non_null
column,,,,
accidents_count_500m,Int64,612,100.00,0
irve_count_500m,Int64,612,100.00,0
pedestrian_network_distance_m,Int64,612,100.00,0
transit_departures_peak_hour,Int64,612,100.00,0
city,Int64,612,100.00,0
public_parking_spaces_300m,Int64,555,90.69,24
amenities_count_500m_straight_line,Int64,0,0.00,229
latitude,Float64,0,0.00,606
station_name,string,0,0.00,612


Fully duplicated rows: 0
Missing station identifiers: 0
Rows with repeated station identifiers: 0


## Supprimer les colonnes avec au moins 90 % de valeurs manquantes

Une colonne est supprimée de la copie de travail `df` lorsque sa proportion de cellules manquantes atteint **90 % ou plus**. Les zéros sont des valeurs renseignées et ne comptent pas comme manquants. Le tableau ci-dessous indique les colonnes retirées et leur taux de données manquantes.

`df_raw` et le CSV source restent intacts. L’identifiant `station_uid` et les colonnes de traçabilité sont conservés s’ils sont renseignés en dessous de ce seuil.

In [3]:
missing_threshold = 0.90
missing_ratio = df.isna().mean()
columns_to_drop = missing_ratio.index[missing_ratio >= missing_threshold].tolist()
dropped_columns = pd.DataFrame({
    "missing_count": df[columns_to_drop].isna().sum(),
    "missing_percent": (missing_ratio[columns_to_drop] * 100).round(2),
}).rename_axis("column").sort_values("missing_percent", ascending=False)

df = df.drop(columns=columns_to_drop)
display(dropped_columns)
print(f"Dropped columns (at least {missing_threshold:.0%} missing): {len(columns_to_drop)}")
print(f"Remaining dataset: {len(df):,} stations and {len(df.columns)} columns")
display(df.head())

,missing_count,missing_percent
column,,
irve_count_500m,612,100.00
transit_departures_peak_hour,612,100.00
pedestrian_network_distance_m,612,100.00
accidents_count_500m,612,100.00
city,612,100.00
public_parking_spaces_300m,555,90.69


Dropped columns (at least 90% missing): 6
Remaining dataset: 612 stations and 44 columns


,station_uid,station_name,latitude,longitude,amenities_count_500m,amenities_count_500m_straight_line,amenities_unreachable_or_unsnapped_500m,bike_parking_count_300m,bike_parking_count_300m_straight_line,bike_parking_unreachable_or_unsnapped_300m,...,radius_1000m_inside_ign_region,boundary_coverage_status,departement,region,reseau,capacity,num_vehicles_available,is_renting,extracted_at,station_reference_match
0,offre-et-disponibilite-temps-reel-du-service-c...,Lyon 2ème - Parking LPA République,45.760189,4.836602,56,323,177,5,36,49,...,True,within_region_conservative_check,Rhône,Auvergne-Rhône-Alpes,Métropole de Lyon,3,3,True,2026-10-19 14:43:54.580579+00:00,both
1,offre-et-disponibilite-temps-reel-du-service-c...,Lyon 6ème - Bugeaud - Garibaldi,45.766689,4.851268,1,326,0,4,61,0,...,True,within_region_conservative_check,Rhône,Auvergne-Rhône-Alpes,Métropole de Lyon,2,2,True,2026-10-19 14:43:54.580579+00:00,both
2,offre-et-disponibilite-temps-reel-du-service-c...,Sainte-Foy - Place St Luc,45.749405,4.8011,9,29,11,1,8,22,...,True,within_region_conservative_check,Rhône,Auvergne-Rhône-Alpes,Métropole de Lyon,1,1,True,2026-10-19 14:43:54.580579+00:00,both
3,offre-et-disponibilite-temps-reel-du-service-c...,Lyon 2ème - Charité - Auguste Comte,45.754833,4.83118,4,262,0,1,53,0,...,True,within_region_conservative_check,Rhône,Auvergne-Rhône-Alpes,Métropole de Lyon,2,1,True,2026-10-19 14:43:54.580579+00:00,both
4,offre-et-disponibilite-temps-reel-du-service-c...,Lyon 5ème - Quai Pierre Scize,45.766705,4.820361,0,42,0,1,25,0,...,True,within_region_conservative_check,Rhône,Auvergne-Rhône-Alpes,Métropole de Lyon,3,3,True,2026-10-19 14:43:54.580579+00:00,both


## Statistiques des variables numériques

Ces statistiques portent sur les données après suppression des colonnes ayant au moins 90 % de valeurs manquantes. Pour chaque variable numérique : moyenne, écart-type d’échantillon (ddof = 1), minimum, premier quartile (25 %), médiane (50 %), troisième quartile (75 %), maximum, nombre et pourcentage de valeurs manquantes. Les calculs excluent les valeurs manquantes sans les remplacer par zéro. Les booléens sont étudiés dans le tableau qualitatif suivant.

In [4]:
numeric_data = df.select_dtypes(include="number")
numeric_stats = numeric_data.describe(percentiles=[0.25, 0.5, 0.75]).T
numeric_stats = numeric_stats.rename(columns={
    "count": "Effectif renseigné", "mean": "Moyenne", "std": "Écart-type",
    "min": "Minimum", "25%": "Q1 (25 %)", "50%": "Médiane (50 %)",
    "75%": "Q3 (75 %)", "max": "Maximum",
})
numeric_stats["Valeurs manquantes"] = numeric_data.isna().sum()
numeric_stats["Valeurs manquantes (%)"] = numeric_data.isna().mean() * 100
numeric_stats.index.name = "Variable"
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(numeric_stats.round(2))

,Effectif renseigné,Moyenne,Écart-type,Minimum,Q1 (25 %),Médiane (50 %),Q3 (75 %),Maximum,Valeurs manquantes,Valeurs manquantes (%)
Variable,,,,,,,,,,
latitude,612.0,45.54,0.33,44.82,45.19,45.58,45.77,46.37,0,0.0
longitude,612.0,5.42,0.56,3.2,4.85,5.7,5.79,6.71,0,0.0
amenities_count_500m,612.0,42.05,54.9,0.0,9.0,22.0,49.0,302.0,0,0.0
amenities_count_500m_straight_line,612.0,103.89,108.34,0.0,25.75,59.0,143.5,623.0,0,0.0
amenities_unreachable_or_unsnapped_500m,612.0,109.8,138.79,0.0,12.75,52.5,150.25,660.0,0,0.0
bike_parking_count_300m,612.0,16.93,22.15,0.0,3.0,9.0,20.0,109.0,0,0.0
bike_parking_count_300m_straight_line,612.0,38.04,37.89,0.0,10.0,25.0,53.25,176.0,0,0.0
bike_parking_unreachable_or_unsnapped_300m,612.0,147.78,221.31,0.0,12.0,55.0,156.0,952.0,0,0.0
tourism_sites_count_1km,612.0,31.03,53.45,0.0,1.0,7.0,33.0,230.0,0,0.0


## Statistiques des variables qualitatives et booléennes

Le résumé indique les modalités distinctes renseignées, toutes les valeurs dominantes en cas d’égalité, leur effectif et leur fréquence parmi les observations renseignées. Le tableau détaillé donne ensuite **toutes les modalités de chaque variable**, y compris les noms et identifiants de stations, avec leurs effectifs et fréquences. Les valeurs manquantes sont signalées séparément.

La fréquence renseignée utilise comme dénominateur les valeurs non manquantes de la variable. La fréquence totale utilise toutes les stations ; une ligne manquante éventuelle n’a pas de fréquence renseignée.

In [5]:
categorical_data = df.select_dtypes(exclude="number")
summary_rows = []
frequency_frames = []
for column in categorical_data:
    series = categorical_data[column]
    counts = series.value_counts(dropna=True)
    non_null_count = int(series.notna().sum())
    top_count = int(counts.max()) if not counts.empty else 0
    modes = counts.index[counts.eq(top_count)].tolist() if not counts.empty else []
    summary_rows.append({
        "Variable": column,
        "Effectif renseigné": non_null_count,
        "Valeurs manquantes": int(series.isna().sum()),
        "Modalités distinctes": len(counts),
        "Valeur(s) dominante(s)": modes,
        "Effectif par modalité dominante": top_count,
        "Fréquence dominante (%)": 100 * top_count / non_null_count if non_null_count else float("nan"),
    })
    frequencies = series.value_counts(dropna=False).rename_axis("Modalité").reset_index(name="Effectif")
    frequencies.insert(0, "Variable", column)
    frequencies["Valeur manquante"] = frequencies["Modalité"].isna()
    frequencies["Fréquence totale (%)"] = frequencies["Effectif"] / len(df) * 100
    frequencies["Fréquence renseignée (%)"] = frequencies["Effectif"] / non_null_count * 100 if non_null_count else float("nan")
    frequencies.loc[frequencies["Valeur manquante"], "Fréquence renseignée (%)"] = float("nan")
    frequency_frames.append(frequencies)

categorical_stats = pd.DataFrame(summary_rows).set_index("Variable")
category_frequencies = pd.concat(frequency_frames, ignore_index=True)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(categorical_stats.round(2))
    for column, frequencies in category_frequencies.groupby("Variable", sort=False):
        print(f"Variable: {column}")
        display(frequencies.drop(columns="Variable").reset_index(drop=True).round(2))

assert len(numeric_stats) + len(categorical_stats) == df.shape[1]
for column, frequencies in category_frequencies.groupby("Variable", sort=False):
    assert frequencies["Effectif"].sum() == len(df)
    assert abs(frequencies["Fréquence totale (%)"].sum() - 100) < 1e-8
print(f"Described {len(numeric_stats)} numeric variables and {len(categorical_stats)} categorical/boolean variables.")

,Effectif renseigné,Valeurs manquantes,Modalités distinctes,Valeur(s) dominante(s),Effectif par modalité dominante,Fréquence dominante (%)
Variable,,,,,,
station_uid,612,0,612,"[offre-et-disponibilite-temps-reel-du-service-citiz-en-autopartage-metropole-de-lyon:1115, offre-et-disponibilite-te...",1,0.16
station_name,612,0,612,"[Lyon 2ème - Parking LPA République, Lyon 6ème - Bugeaud - Garibaldi, Sainte-Foy - Place St Luc, Lyon 2ème - Charité...",1,0.16
walking_network_status,612,0,1,[matched],612,100.00
public_parking_spaces_300m_status,612,0,2,[unknown_access_or_capacity],555,90.69
irve_count_500m_status,612,0,1,[charging_units_not_defined_osm_sites_only],612,100.00
transit_departures_peak_hour_status,612,0,1,[missing_gtfs_service_date_and_peak_window],612,100.00
pedestrian_network_distance_m_status,612,0,1,[missing_origins_destinations_and_regional_graph],612,100.00
accidents_count_500m_status,612,0,1,[missing_accident_data_and_period],612,100.00
osm_count_status,612,0,1,[observed_mapped_objects_not_exhaustive],612,100.00


Variable: station_uid


,Modalité,Effectif,Valeur manquante,Fréquence totale (%),Fréquence renseignée (%)
0,offre-et-disponibilite-temps-reel-du-service-citiz-en-autopartage-metropole-de-lyon:1115,1,False,0.16,0.16
1,offre-et-disponibilite-temps-reel-du-service-citiz-en-autopartage-metropole-de-lyon:1116,1,False,0.16,0.16
2,offre-et-disponibilite-temps-reel-du-service-citiz-en-autopartage-metropole-de-lyon:1117,1,False,0.16,0.16
3,offre-et-disponibilite-temps-reel-du-service-citiz-en-autopartage-metropole-de-lyon:1118,1,False,0.16,0.16
4,offre-et-disponibilite-temps-reel-du-service-citiz-en-autopartage-metropole-de-lyon:1119,1,False,0.16,0.16
5,offre-et-disponibilite-temps-reel-du-service-citiz-en-autopartage-metropole-de-lyon:1120,1,False,0.16,0.16
6,offre-et-disponibilite-temps-reel-du-service-citiz-en-autopartage-metropole-de-lyon:1121,1,False,0.16,0.16
7,offre-et-disponibilite-temps-reel-du-service-citiz-en-autopartage-metropole-de-lyon:1122,1,False,0.16,0.16
8,offre-et-disponibilite-temps-reel-du-service-citiz-en-autopartage-metropole-de-lyon:1123,1,False,0.16,0.16
9,offre-et-disponibilite-temps-reel-du-service-citiz-en-autopartage-metropole-de-lyon:1124,1,False,0.16,0.16


Variable: station_name


,Modalité,Effectif,Valeur manquante,Fréquence totale (%),Fréquence renseignée (%)
0,Lyon 2ème - Parking LPA République,1,False,0.16,0.16
1,Lyon 6ème - Bugeaud - Garibaldi,1,False,0.16,0.16
2,Sainte-Foy - Place St Luc,1,False,0.16,0.16
3,Lyon 2ème - Charité - Auguste Comte,1,False,0.16,0.16
4,Lyon 5ème - Quai Pierre Scize,1,False,0.16,0.16
5,Lyon 5ème - Valdo - Witkowski,1,False,0.16,0.16
6,Lyon 5ème - Radisson - Jaricot,1,False,0.16,0.16
7,Lyon 5ème - Place Gerson,1,False,0.16,0.16
8,Lyon 5ème - Joliot Curie,1,False,0.16,0.16
9,Lyon 5ème - Charcot-Castors,1,False,0.16,0.16


...

Variable: extracted_at


,Modalité,Effectif,Valeur manquante,Fréquence totale (%),Fréquence renseignée (%)
0,2026-10-19 14:43:54.580579+00:00,612,False,100.0,100.0


Variable: station_reference_match


,Modalité,Effectif,Valeur manquante,Fréquence totale (%),Fréquence renseignée (%)
0,both,612,False,100.0,100.0


Described 24 numeric variables and 20 categorical/boolean variables.
